# Credit Risk Panel Data — Exploratory Data Analysis

**Objective:** Profile a high-dimensional, longitudinal customer dataset (`gb` = default / bad flag) and establish modeling guardrails before training.

**Dataset:** `train_df.csv` — 26,824 rows × 554 columns, 5,243 unique entities (`id`), severe class imbalance (~2.2% positive).

---

## Narrative Structure

1. [Data Overview](#1-data-overview)
2. [Missing Values & Data Quality](#2-missing-values--data-quality)
3. [Outliers & Distribution Shape](#3-outliers--distribution-shape)
4. [Feature Distributions](#4-feature-distributions)
5. [Bivariate & Multivariate Relationships](#5-bivariate--multivariate-relationships)
6. [Panel Structure, Leakage & Longitudinal Dynamics](#6-panel-structure-leakage--longitudinal-dynamics)
7. [Conclusions & Roadmap for Model Training](#7-conclusions--roadmap-for-model-training)


## Setup

Imports, logging, plotting theme, and reusable analysis functions from `eda_utils.py`.

In [ ]:
import logging
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

import eda_utils as eu

warnings.filterwarnings("ignore", category=FutureWarning)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)
logger = logging.getLogger("eda")

DATA_PATH = "train_df.csv"
TARGET_COL = "gb"
ID_COL = "id"

eu.configure_plotting()
%matplotlib inline


## 1. Data Overview

We load the data, validate schema, and summarize scale, target balance, and panel structure.

In [ ]:
df = eu.load_data(DATA_PATH, target_col=TARGET_COL, id_col=ID_COL)
groups = eu.get_feature_groups(df, id_col=ID_COL, target_col=TARGET_COL)
profile = eu.profile_dataset(df, groups)

summary = pd.DataFrame(
    {
        "Metric": [
            "Rows",
            "Columns",
            "Unique IDs",
            "Mean rows per ID",
            "Numeric features",
            "Categorical features",
            "Positive rate (gb=1)",
            "Class imbalance ratio (0:1)",
            "Duplicate rows",
        ],
        "Value": [
            f"{profile.n_rows:,}",
            profile.n_cols,
            f"{profile.n_ids:,}",
            f"{profile.rows_per_id.mean():.2f}",
            len(groups.numeric),
            len(groups.categorical),
            f"{profile.target_rate:.2%}",
            f"{profile.imbalance_ratio:.1f}:1",
            df.duplicated().sum(),
        ],
    }
)
display(summary)

fig = eu.plot_target_distribution(profile, target_col=TARGET_COL)
plt.show()


### Interpretation — Data Overview

- **Panel, not i.i.d.:** ~5.1 observations per `id` on average (max 9) means each row is a *snapshot in an entity's history*, not an independent customer.
- **Severe imbalance:** Only **~2.21%** positives (`gb=1`). Accuracy near 98% is trivial; models must optimize **ranking** metrics (ROC-AUC, PR-AUC) and use **class weights** or resampling.
- **High dimensionality:** 416 numeric + 135 categorical engineered features suggest a rich credit-bureau / behavioral feature store typical of retail risk scoring.
- **No duplicate rows:** Row-level uniqueness is intact; redundancy lives at the *column* level (addressed later).


## 2. Missing Values & Data Quality

Missingness is structural — many columns are sparse or entirely empty. We quantify patterns and test whether *being missing* predicts the target (MNAR signal).

In [ ]:
num_df = df[groups.numeric]
cat_df = df[groups.categorical]

missing_all = eu.missingness_summary(df, groups.all_features)
missing_num = eu.missingness_summary(df, groups.numeric)

empty_cols = missing_num[missing_num["missing_pct"] == 100]
heavy_missing = missing_num[(missing_num["missing_pct"] >= 50) & (missing_num["missing_pct"] < 100)]

quality = pd.DataFrame(
    {
        "Check": [
            "100% empty numeric columns",
            "Numeric columns with ≥50% missing",
            "Global mean missing rate (all features)",
        ],
        "Count / Value": [
            len(empty_cols),
            len(heavy_missing),
            f"{df[groups.all_features].isnull().mean().mean() * 100:.2f}%",
        ],
    }
)
display(quality)
display(empty_cols.head(12))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
eu.plot_missingness_distribution(missing_num, ax=axes[0])
eu.plot_top_missing_features(missing_num, top_n=15, ax=axes[1])
plt.tight_layout()
plt.show()


In [ ]:
dup_groups = eu.detect_duplicate_column_groups(df, groups.all_features)
print(f"Duplicate column groups (≥{eu.MIN_VALID_VALUES} valid values): {len(dup_groups)}")
for canonical, twins in list(dup_groups.items())[:5]:
    print(f"  {canonical} -> {twins}")

high_corr_n, high_corr_sample = eu.count_high_correlation_pairs(df, groups.numeric)
print(f"Highly correlated numeric pairs (|r| > {eu.CORR_THRESHOLD}): {high_corr_n}")
display(high_corr_sample.head(10))


In [ ]:
chi2_missing = eu.chi2_missingness_vs_target(
    df,
    groups.numeric,
    target_col=TARGET_COL,
    min_missing_pct=10.0,
)
sig_missing = chi2_missing[chi2_missing["p_value"] < 0.05]
print(f"Numeric features where missingness depends on target (p < 0.05): {len(sig_missing)}")
display(sig_missing.head(10))


### Interpretation — Missing Values & Redundancy

- **12 columns are 100% empty** (e.g. `num_23`, `num_66`) — drop unconditionally; they carry zero information.
- **~153 numeric features are ≥50% missing** — sparsity is a first-class modeling axis. Tree models (XGBoost, CatBoost, LightGBM) handle NaNs natively; linear models need **imputation + missing indicators**.
- **MNAR (Missing Not At Random):** Dozens of features show statistically significant association between *missingness* and `gb` (e.g. `num_99`, `num_326` with extreme p-values). In credit risk, missingness often means *"product not held"* or *"not yet scored"* — a legitimate risk signal. **Do not impute blindly without preserving missing flags** for linear models.
- **Massive redundancy:** Identical column groups and **700+ highly correlated pairs** (`|r| > 0.95`) will destabilize logistic regression and inflate variance. Tree ensembles are more robust but still benefit from deduplication for interpretability and training speed.
- **Business implication:** Feature stores were likely assembled from multiple sources with overlapping definitions; production pipelines should enforce uniqueness at ingestion.


## 3. Outliers & Distribution Shape

Heavy tails and skewness are common in financial ratios. We profile IQR-based outlier rates and variance degeneracy.

In [ ]:
outlier_rates = eu.iqr_outlier_rates(num_df)
high_outlier_cols = outlier_rates[outlier_rates > eu.IQR_OUTLIER_THRESHOLD].dropna()
low_var_cols = eu.low_variance_columns(num_df)
skew = eu.numeric_skewness(num_df)

shape_stats = pd.DataFrame(
    {
        "Metric": [
            "Features with >5% IQR outliers",
            "Quasi-constant features (var < 0.01)",
            "Features with |skew| > 2",
            "Median skewness (active numerics)",
        ],
        "Value": [
            len(high_outlier_cols),
            len(low_var_cols),
            int((skew.abs() > 2).sum()),
            f"{skew.median():.2f}",
        ],
    }
)
display(shape_stats)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(outlier_rates.dropna(), bins=40, kde=True, color=eu.PALETTE["primary"], ax=axes[0])
axes[0].axvline(eu.IQR_OUTLIER_THRESHOLD, color=eu.PALETTE["accent"], linestyle="--", label="5% threshold")
axes[0].set_title("Distribution of Per-Feature Outlier Rates (IQR)")
axes[0].set_xlabel("Fraction of IQR outliers")
axes[0].legend()

sns.histplot(skew.dropna(), bins=40, kde=True, color=eu.PALETTE["secondary"], ax=axes[1])
axes[1].axvline(2, color=eu.PALETTE["accent"], linestyle="--")
axes[1].axvline(-2, color=eu.PALETTE["accent"], linestyle="--")
axes[1].set_title("Distribution of Numeric Skewness")
axes[1].set_xlabel("Skewness")
plt.tight_layout()
plt.show()


### Interpretation — Outliers & Skewness

- **274+ features exceed 5% IQR outliers** — expected for amounts, utilization, and delinquency counters. **Tree-based models are scale-invariant** and handle extremes via splits; **linear models and neural nets require robust scaling** (e.g. `RobustScaler`) or winsorization.
- **Median skewness ≈ 9.7** with **313 features |skew| > 2** — distributions are far from Gaussian. Log transforms or Yeo-Johnson may help linear models but are optional for boosting.
- **7 quasi-constant numerics** — candidates for removal (near-zero information gain).
- **Anomaly note:** Extreme values may reflect data entry errors *or* genuine distressed borrowers; use **entity-grouped** validation before treating outliers as noise.


## 4. Feature Distributions

Categorical cardinality, numeric separability (Mann-Whitney), and sample KDE overlays for top discriminators.

In [ ]:
cardinality = eu.categorical_cardinality(cat_df)
high_card = cardinality[cardinality > 50]

cat_profile = pd.DataFrame(
    {
        "Metric": [
            "Categorical features",
            "Median cardinality",
            "High-cardinality features (>50 levels)",
        ],
        "Value": [
            len(groups.categorical),
            f"{cardinality.median():.0f}",
            len(high_card),
        ],
    }
)
display(cat_profile)
display(high_card.head())

fig, ax = plt.subplots(figsize=(10, 5))
eu.plot_cardinality_distribution(cardinality, ax=ax)
plt.tight_layout()
plt.show()

if len(high_card) > 0:
    example = high_card.index[0]
    print(f"Top levels for high-cardinality feature `{example}`:")
    display(df[example].value_counts(normalize=True).head(8).to_frame("proportion"))


In [ ]:
mw_results = eu.mannwhitney_by_target(df, groups.numeric, target_col=TARGET_COL)
sig_mw = mw_results[mw_results["significant"]]
print(f"Significant numeric features (Mann-Whitney, p < 0.05): {len(sig_mw)} / {len(mw_results)}")
display(sig_mw.head(10))

top_features = sig_mw["feature"].head(6).tolist()
fig = eu.plot_numeric_distribution_sample(df, top_features, target_col=TARGET_COL)
plt.show()


### Interpretation — Feature Distributions

- **Low median cardinality (3)** — most `cat_*` columns are coarse bins (likely score bands or product flags). **One-hot encoding is feasible** for the majority.
- **5 high-cardinality categoricals** (e.g. `cat_19` with 50+ levels, dominant level ~52%) — one-hot expansion is expensive; prefer **target/frequency encoding**, **CatBoost native categoricals**, or **hashing** with regularization.
- **233 / 375 tested numerics differ significantly across classes** — strong univariate signal exists; top features (`num_105`, `num_197`, `num_43`) show clear class separation in KDE plots.
- **Business read:** Separating distributions suggest these features capture utilization, delinquency depth, or bureau scores — classic drivers of default risk.


## 5. Bivariate & Multivariate Relationships

Correlation structure, categorical-target association, and entity-level activity vs. risk.

In [ ]:
# Correlation heatmap on a representative low-dimensional subset (top variance numerics)
active_num = [c for c in groups.numeric if df[c].notna().sum() >= 500]
variances = df[active_num].var().sort_values(ascending=False)
heatmap_cols = variances.head(20).index.tolist()

corr_sub = df[heatmap_cols].corr()
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    corr_sub,
    cmap="RdBu_r",
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    ax=ax,
)
ax.set_title("Correlation Matrix — Top 20 High-Variance Numeric Features")
plt.tight_layout()
plt.show()


In [ ]:
# Cramér's V for categoricals (sample top 15 by cardinality for runtime)
cat_sample = cardinality.head(30).index.tolist()
cramers = []
for col in cat_sample:
    v = eu.cramers_v_correlation(df[col].fillna(-1), df[TARGET_COL])
    cramers.append({"feature": col, "cramers_v": v, "nunique": df[col].nunique()})
cramers_df = pd.DataFrame(cramers).sort_values("cramers_v", ascending=False)
display(cramers_df.head(10))

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=cramers_df.head(12), x="cramers_v", y="feature", palette="Blues_d", ax=ax)
ax.set_title("Cramér's V — Categorical Association with Target (Top 12)")
ax.set_xlabel("Cramér's V")
plt.tight_layout()
plt.show()


In [ ]:
activity_risk = eu.id_activity_risk_table(df, ID_COL, TARGET_COL)
display(activity_risk)

fig, ax = plt.subplots(figsize=(9, 5))
activity_risk["bad_rate_pct"].plot(kind="bar", color=eu.PALETTE["primary"], ax=ax)
ax.set_ylabel("Bad rate (%)")
ax.set_xlabel("Activity quintile (rows per ID)")
ax.set_title("Default Rate vs. Entity Activity (Rows per ID)")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()


### Interpretation — Multivariate Relationships

- **Multicollinearity clusters** visible in the heatmap — many `num_*` blocks move together (likely derived from the same bureau attribute). For **logistic regression**, apply **VIF pruning** or **PCA** on correlated blocks; for **boosting**, optional pruning improves speed.
- **Categorical-target association** varies; low-cardinality cats with high Cramér's V are prime candidates for tree splits.
- **Activity–risk gradient:** Entities with **fewer rows (1–2 snapshots)** show **~5.0%** bad rate vs **~1.2%** for the most active quintile. This may reflect **shorter observation windows** (less time to cure) or **newer / higher-risk entrants** — important for **group-aware** and potentially **time-aware** validation.
- **Collinearity + panel structure** together imply that global correlation estimates mix **between-entity** and **within-entity** variation; the static/dynamic split (next section) clarifies which relationships are safe to exploit.


## 6. Panel Structure, Leakage & Longitudinal Dynamics

Anti-leakage validation: target stability per ID, monotonic timeline features, static vs. dynamic numerics, and macro drift.

In [ ]:
total_ids, dynamic_ids = eu.target_stability_per_id(df, ID_COL, TARGET_COL)
print(f"IDs with changing target: {dynamic_ids} / {total_ids} ({dynamic_ids/total_ids:.2%})")

mono = eu.monotonic_feature_scan(df, ID_COL, groups.numeric)
print(f"Potential timeline / sequence features (≥80% monotonic per ID): {len(mono)}")
display(mono.head(8))

static_dyn = eu.static_vs_dynamic_features(df, ID_COL, groups.numeric)
print(static_dyn["type"].value_counts())

drift = eu.global_position_drift(df, TARGET_COL)
display(drift)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

static_dyn["variance_ratio"].plot(
    kind="hist",
    bins=50,
    color=eu.PALETTE["primary"],
    ax=axes[0],
)
axes[0].axvline(0.05, color=eu.PALETTE["accent"], linestyle="--", label="Static threshold (5%)")
axes[0].set_title("Within-ID / Global Variance Ratio")
axes[0].set_xlabel("Variance ratio")
axes[0].legend()

drift["bad_rate_pct"].plot(kind="bar", color=eu.PALETTE["secondary"], ax=axes[1])
axes[1].set_ylabel("Bad rate (%)")
axes[1].set_title("Target Rate by File-Order Quartile")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()


### Interpretation — Panel & Leakage

| Finding | Implication |
|--------|-------------|
| **13 / 5,243 IDs** change `gb` across rows | Target is **almost always entity-static**; treat as customer-level label with rare transitions |
| **~240 monotonic numerics** per ID | Rows are **time-ordered snapshots**; row index is a proxy timeline |
| **~47.5% static numerics** | Half the signal is **entity fingerprint**; half is **behavioral drift** |
| **Flat macro drift** (2.08–2.50% bad rate by file chunk) | No strong **global temporal shift** in this file |

**Critical modeling rule:** Never use `train_test_split` without grouping on `id`. Random splits leak static features and future snapshots → **inflated offline AUC, collapsed production performance**.

**Recommended validation:** `GroupKFold` / `StratifiedGroupKFold` on `id` (stratify by entity-level target). If predicting *next* snapshot, add **time-ordered split within ID**.


## 7. Conclusions & Roadmap for Model Training

### Executive Summary

This is a **high-dimensional, sparse, longitudinal credit-risk panel** with **severe class imbalance** and **rich MNAR missingness**. Univariate tests confirm abundant signal, but **redundancy and panel leakage** dominate the modeling risk — not raw predictive power.

---

### Feature Handling

| Action | Features | Rationale |
|--------|----------|------------|
| **Drop** | 12 × 100% empty numerics; exact duplicate columns; quasi-constant (`var < 0.01`) | Zero or redundant information |
| **Keep + indicator** | High-missing numerics with significant missingness–target association | MNAR signal for linear models |
| **Engineer** | Lag / delta on dynamic monotonic features (`num_2`, `num_4`, …); row index per `id`; aggregation (min/max/mean/std) per entity | Captures velocity & entity-level summaries |
| **Encode** | Low-cardinality `cat_*` → target encoding or OHE; high-cardinality (`cat_19`, …) → frequency / CatBoost native | Controls dimensionality |
| **Transform** | Skewed numerics → Yeo-Johnson or log1p for linear/NN; **RobustScaler** for GLM/NN | Stabilize linear optimization |
| **Prune (linear track)** | One from each duplicate group; drop one of each pair with \|r\| > 0.95 | Multicollinearity control |

---

### Data Splitting Strategy

1. **Primary:** `StratifiedGroupKFold(n_splits=5, groups=id, y=gb)` — ensures no `id` appears in both train and validation; stratification preserves ~2.2% positive rate per fold.
2. **Alternative (deployment simulation):** Hold out 20% of IDs entirely as a **locked test set**; never tune on it.
3. **If modeling sequential default:** Sort by within-`id` row order; use **expanding-window** or **last-snapshot-only** labels to avoid predicting the past from the future.

**Why not random K-Fold?** Static entity fingerprints + temporal sequences → **identity memorization** and **look-ahead bias**.

---

### Recommended Algorithms

| Algorithm | Fit for this data | Notes |
|-----------|---------------------|-------|
| **CatBoost / LightGBM / XGBoost** | ★★★★★ | Native missing + categorical handling; robust to outliers & collinearity |
| **Random Forest** | ★★★☆☆ | Strong baseline; slower at 550 features; less efficient with sparse cats |
| **Logistic Regression (L1/L2)** | ★★☆☆☆ | Needs dedup, scaling, missing indicators; good for interpretability |
| **Neural network (embeddings)** | ★★★☆☆ | Viable for high-cardinality cats; needs careful grouped validation |

**First production candidate:** **CatBoostClassifier** (`auto_class_weights='Balanced'`, `eval_metric='AUC'`) with `id` as group key in CV.

---

### Metrics to Optimize

| Metric | Role |
|--------|------|
| **PR-AUC (Average Precision)** | **Primary** — sensitive to rare positives (~2.2%) |
| **ROC-AUC** | Secondary — ranking quality across thresholds |
| **F1 @ business threshold** | Tie to operational cutoff once cost matrix is known |
| **Precision@K / Lift@10%** | Align with campaign capacity (top-decile targeting) |
| **Calibration (Brier, reliability curve)** | If scores feed pricing or provisioning |

**Do not optimize accuracy.** Report **confusion matrix at chosen threshold** only after threshold tuning on validation.

---

### Suggested Next Steps (Implementation Order)

1. Build **sklearn Pipeline**: drop empty → dedupe → optional missing indicators → model.
2. Implement **grouped CV** with out-of-fold target encoding for categoricals.
3. Train **CatBoost baseline**; capture SHAP on top 30 features for stakeholder narrative.
4. Add **entity aggregations** + **delta features**; compare PR-AUC lift.
5. Lock **hold-out ID set** and document expected production degradation (~5–15% AUC drop vs. random split is normal and healthy).

---

*Analysis generated with modular `eda_utils.py` — suitable for import into training pipelines and scheduled EDA jobs.*
